# RAG Demo — Pipeline Walkthrough

A minimal, modular Retrieval-Augmented Generation pipeline. Each stage is a standalone module that reads from one folder under `data/` and writes to the next, so you can run, swap, or upgrade each block independently.

**Stages** (built incrementally):
1. **Document processing** — load raw files, extract to markdown ← *covered below*
2. **Chunker** — split markdown into overlapping chunks ← *covered below*
3. **Embedder** — turn chunks into vectors (local or OpenAI) ← *covered below*
4. **Retriever** — top-k chunks for a query ← *covered below*
5. **Answerer** — OpenAI generates a cited answer from retrieved context ← *covered below*

### Architecture at a glance

Two pipelines share the same data store: **indexing** runs offline (once per corpus); **query** runs per user question.

![Data flow](../docs/data_flow.svg)

## Stage 1 — Document Processing

- **Input:** `data/raw/` — drop your PDFs, `.md`, or `.txt` files here.
- **Output:** `data/markdown/` — one `.md` per input file, with YAML frontmatter (`title`, `authors`, `source`).
- **Extractor:** `pymupdf` for both PDF text and metadata (see `document_processing.pdf_proc.extract_pdf_basic`); pass-through for native text/markdown.

Two entry points:
- `extract_text(file_path)` — pure function, takes one file, returns a `Document`. Use it to inspect the result for a single file.
- `extract_docs(input_dir, output_dir)` — runs `extract_text` over a folder and writes `.md` (with frontmatter) to disk.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from document_processing import extract_text, extract_docs
from chunker import chunk_doc, chunk_docs
from embedder import LocalEmbedder, ApiEmbedder, ParquetStore, embed_chunk, embed_chunks
from retriever import retrieve
from answerer import answer, DEFAULT_PROMPT

In [3]:
RAW_DIR = ROOT_DIR / "data" / "raw"
MARKDOWN_DIR = ROOT_DIR / "data" / "markdown"
CHUNKS_DIR = ROOT_DIR / "data" / "chunks"
EMBEDDINGS_DIR = ROOT_DIR / "data" / "embeddings"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print("raw dir:       ", RAW_DIR)
print("markdown dir:  ", MARKDOWN_DIR)
print("chunks dir:    ", CHUNKS_DIR)
print("embeddings dir:", EMBEDDINGS_DIR)

raw dir:        c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\raw
markdown dir:   c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\markdown
chunks dir:     c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\chunks
embeddings dir: c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\data\embeddings


### What's in `data/raw/`?

If empty, drop a PDF (or `.md`/`.txt`) in there and re-run.

In [4]:
sorted(path.name for path in RAW_DIR.iterdir() if path.is_file())

['Large Language Models the New Scrum Masters.pdf']

### Single-file extraction (pure function)

Run `extract_text()` on one file and inspect the resulting `Document`.

In [5]:
input_files = [path for path in sorted(RAW_DIR.iterdir()) if path.is_file()]
assert input_files, f"No files in {RAW_DIR}. Drop a PDF/.md/.txt and re-run."

sample_file = input_files[0]
doc = extract_text(file_path=sample_file)

print("source:  ", doc.source.name)
print("metadata:", doc.metadata)
print("length:  ", len(doc.text), "chars")
print()
print("--- first 1500 chars ---")
print(doc.text[:1500])

source:   Large Language Models the New Scrum Masters.pdf
metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
length:   23223 chars

--- first 1500 chars ---
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-papers 
Scholarly Commons Citation 
Scholarly Commons Citation 
Couder, J., & Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from 
https://commons.erau.edu/red-papers/2 
This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile 
Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of 
Scholarly Commons. For more information, please contact comm

### Batch run (stage runner)

`extract_docs` extracts every supported file in `data/raw/` and writes a corresponding `.md` (with YAML frontmatter) to `data/markdown/`.

In [6]:
docs = extract_docs(input_dir=RAW_DIR, output_dir=MARKDOWN_DIR)
for doc in docs:
    print(f"{doc.source.name:40s} -> {len(doc.text):>8} chars  {doc.metadata}")

Large Language Models the New Scrum Masters.pdf ->    23223 chars  {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}


### Inspect the output folder

In [7]:
for path in sorted(MARKDOWN_DIR.iterdir()):
    if path.is_file() and path.suffix == ".md":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

Large Language Models the New Scrum Masters [basic].md    23428 bytes
Large Language Models the New Scrum Masters [docling].md    33204 bytes
Large Language Models the New Scrum Masters [pymupdf4llm].md    24027 bytes
Large Language Models the New Scrum Masters.md    23833 bytes


## Stage 2 — Chunker

- **Input:** `data/markdown/` — `.md` files with YAML frontmatter from stage 1.
- **Output:** `data/chunks/` — one `.jsonl` per source doc, one chunk per line.
- **Strategy:** fixed-size character windows with overlap (default `size=2000`, `overlap=200`).

Two entry points:
- `chunk_doc(doc)` — pure function over an in-memory `Document`.
- `chunk_docs(input_dir, output_dir)` — reads `.md` files via frontmatter, chunks, writes `.jsonl`.

In [8]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

all_chunks = [chunk for doc in docs for chunk in chunk_doc(doc=doc, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)]
print(f"{len(all_chunks)} chunks across {len(docs)} docs (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

first_chunk = all_chunks[0]
print("first chunk:")
print(f"  id:       {first_chunk.id}")
print(f"  source:   {first_chunk.source}")
print(f"  index:    {first_chunk.index}")
print(f"  metadata: {first_chunk.metadata}")
print(f"  text[:300]:\n{first_chunk.text[:300]}")

13 chunks across 1 docs (size=2000, overlap=200)

first chunk:
  id:       Large Language Models the New Scrum Masters#0
  source:   Large Language Models the New Scrum Masters.pdf
  index:    0
  metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
  text[:300]:
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-pap


### Batch run

`chunk_docs` reads every `.md` in `data/markdown/` (frontmatter + body), chunks each, and writes one `.jsonl` per doc to `data/chunks/`.

In [9]:
all_chunks_written = chunk_docs(input_dir=MARKDOWN_DIR, output_dir=CHUNKS_DIR, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f"wrote {len(all_chunks_written)} chunks total (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

for path in sorted(CHUNKS_DIR.iterdir()):
    if path.suffix == ".jsonl":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

AttributeError: module 'frontmatter' has no attribute 'loads'

## Stage 3 — Embedder

- **Input:** `data/chunks/` — `.jsonl` files from stage 2.
- **Output:** whatever the chosen `data_store` writes. The default `ParquetStore` writes one `.parquet` per source doc to `data/embeddings/` with columns `chunk_id, source, text, vector, model, dim`.
- **Embedder backends** (swap with one line):
  - `LocalEmbedder()` — `sentence-transformers/all-MiniLM-L6-v2`, 384-dim, CPU, no API key.
  - `ApiEmbedder()` — OpenAI `text-embedding-3-small`, 1536-dim. Reads `OPENAI_API_KEY` from a `.env` file at the repo root (or accepts an explicit `api_key=` arg).
- **Storage backends:** any class in `embedder/data_stores.py` exposing `add(embeddings, source_stem)` and `load()`. `ParquetStore` is the only one for now; FAISS / Chroma can be added as sibling classes in the same file.

Two entry points (mirror earlier stages):
- `embed_chunk(chunk, embedder)` — pure function, embeds one chunk.
- `embed_chunks(input_dir, embedder, data_store)` — batches the calls per source file, hands each batch to `data_store.add(...)`.

> To use the API backend, create a `.env` at the repo root with `OPENAI_API_KEY=sk-...`. `.env` is gitignored.

In [10]:
# default to local: zero-friction, runs on CPU, no API key required
embedder = LocalEmbedder()
# swap to OpenAI by uncommenting the next line (requires .env with OPENAI_API_KEY)
# embedder = ApiEmbedder()

print(f"backend: {type(embedder).__name__}  model: {embedder.model}  dim: {embedder.dimension}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

backend: LocalEmbedder  model: sentence-transformers/all-MiniLM-L6-v2  dim: 384


c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\embedder\models.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self._st.get_sentence_embedding_dimension()


### Single-chunk embedding (pure function)

In [11]:
sample_chunk = all_chunks[0]
embedding = embed_chunk(chunk=sample_chunk, embedder=embedder)

print(f"chunk_id:   {embedding.chunk_id}")
print(f"source:     {embedding.source}")
print(f"model:      {embedding.model}")
print(f"dim:        {embedding.dim}")
print(f"vector[:8]: {embedding.vector[:8]}")

chunk_id:   Large Language Models the New Scrum Masters#0
source:     Large Language Models the New Scrum Masters.pdf
model:      sentence-transformers/all-MiniLM-L6-v2
dim:        384
vector[:8]: [-0.035487107932567596, -0.07033354043960571, -0.03751732409000397, -0.018446974456310272, -0.0030614102724939585, 0.04107296094298363, -0.12445373833179474, 0.05954372510313988]


### Batch run

`embed_chunks` reads every `.jsonl` in `data/chunks/`, embeds each in batches, and writes one `.parquet` per source doc to `data/embeddings/`.

In [12]:
data_store = ParquetStore(directory=EMBEDDINGS_DIR)
all_embeddings = embed_chunks(input_dir=CHUNKS_DIR, embedder=embedder, data_store=data_store)
print(f"wrote {len(all_embeddings)} embeddings (model={embedder.model}, dim={embedder.dimension})\n")

for path in sorted(EMBEDDINGS_DIR.iterdir()):
    if path.suffix == ".parquet":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

wrote 13 embeddings (model=sentence-transformers/all-MiniLM-L6-v2, dim=384)

Large Language Models the New Scrum Masters.parquet    63371 bytes


### Working with the parquet output

Three patterns the format makes cheap: inspect the schema (parquet is self-describing), read only a subset of columns (no need to pull `text`/`vector` if you don't want them), and load a full row back as a dict.

In [13]:
import pyarrow.parquet as pq

sample_parquet = next(path for path in sorted(EMBEDDINGS_DIR.iterdir()) if path.suffix == ".parquet")

# 1) schema inspection — types and columns live in the file's footer
parquet_file = pq.ParquetFile(source=sample_parquet)
print(f"file:    {sample_parquet.name}")
print(f"rows:    {parquet_file.metadata.num_rows}")
print(f"schema:\n{parquet_file.schema_arrow}\n")

# 2) column projection — pull only the cheap columns, skip text + vector
metadata_columns = pq.read_table(source=sample_parquet, columns=["chunk_id", "source", "model", "dim"])
print("light read (chunk_id, source, model, dim) — first 3 rows:")
print(metadata_columns.slice(offset=0, length=3))

# 3) full row — load one complete record back as a dict
first_row = pq.read_table(source=sample_parquet).slice(offset=0, length=1).to_pylist()[0]
print(f"\nfirst row keys:    {list(first_row.keys())}")
print(f"first vector[:5]:  {first_row['vector'][:5]}")
print(f"first text[:120]:  {first_row['text'][:120]}")

file:    Large Language Models the New Scrum Masters.parquet
rows:    13
schema:
chunk_id: string
source: string
text: string
vector: list<element: double>
  child 0, element: double
model: string
dim: int64

light read (chunk_id, source, model, dim) — first 3 rows:
pyarrow.Table
chunk_id: string
source: string
model: string
dim: int64
----
chunk_id: [["Large Language Models the New Scrum Masters#0","Large Language Models the New Scrum Masters#1","Large Language Models the New Scrum Masters#2"]]
source: [["Large Language Models the New Scrum Masters.md","Large Language Models the New Scrum Masters.md","Large Language Models the New Scrum Masters.md"]]
model: [["sentence-transformers/all-MiniLM-L6-v2","sentence-transformers/all-MiniLM-L6-v2","sentence-transformers/all-MiniLM-L6-v2"]]
dim: [[384,384,384]]

first row keys:    ['chunk_id', 'source', 'text', 'vector', 'model', 'dim']
first vector[:5]:  [-0.03548711910843849, -0.0703335627913475, -0.037517327815294266, -0.018446967005729675,

## Stage 4 — Retriever

- **Input:** the same `data_store` produced by stage 3 (any class with `load() -> list[Embedding]`).
- **Output:** in-memory `list[RetrievalResult]` returned to the caller — no on-disk artifact, since results are query-time.
- **Strategy:** embed the query with the *same* embedder used at index time, score every stored vector by cosine similarity (numpy matmul), return the top-k.

One entry point:
- `retrieve(query, embedder, data_store, top_k)` — returns `RetrievalResult(chunk_id, source, text, score)` for the best matches.

> The embedder must match the one used to populate the store. Mixing models silently degrades quality — that's why each parquet row records `model` and `dim`.

In [14]:
query = "How can large language models support a Scrum Master?"
results = retrieve(query=query, embedder=embedder, data_store=data_store, top_k=3)

print(f"query: {query}\n")
for rank, result in enumerate(results, start=1):
    print(f"#{rank}  score={result.score:.4f}  {result.chunk_id}  ({result.source})")
    print(f"    {result.text[:240].replace(chr(10), ' ')}...")
    print()

query: How can large language models support a Scrum Master?

#1  score=0.6981  Large Language Models the New Scrum Masters#3  (Large Language Models the New Scrum Masters.md)
     which consists of professionals who work together to deliver increments of potentially shippable product  functionality during each Sprint. Traditionally, Scrum Masters have been human facilitators, guiding  teams through the intricacies o...

#2  score=0.6641  Large Language Models the New Scrum Masters#0  (Large Language Models the New Scrum Masters.md)
    Papers  RED Innovation: Using Scrum to Develop an  Agile Department  2024  Large Language Models, the New Scrum Masters  Large Language Models, the New Scrum Masters  Juan Couder  ortizcoj@my.erau.edu  Omar Ochoa  ochoao@erau.edu  Follow th...

#3  score=0.6604  Large Language Models the New Scrum Masters#1  (Large Language Models the New Scrum Masters.md)
    s. Scrum is a framework used within  agile software development but has found application in v

In [15]:
import numpy as np

# Same query, but unrolled step by step — the math `retrieve()` does internally.

# 1) embed the query with the same backend used at index time
demo_query = "How can large language models support a Scrum Master?"
[query_vector] = embedder([demo_query])
print(f"query vector dim: {len(query_vector)}")
print(f"first 5 values:   {[round(x, 4) for x in query_vector[:5]]}\n")

# 2) load every stored embedding via the data_store
stored_embeddings = data_store.load()
print(f"stored embeddings: {len(stored_embeddings)} chunks (dim={stored_embeddings[0].dim})\n")

# 3) stack into an (N, dim) numpy matrix; cast query to float32 too
matrix = np.array([emb.vector for emb in stored_embeddings], dtype=np.float32)
query_arr = np.array(query_vector, dtype=np.float32)
print(f"matrix shape: {matrix.shape}")
print(f"query shape:  {query_arr.shape}\n")

# 4) cosine similarity, in three pieces:
#       dots   = matrix @ query     → (N,) per-row dot product
#       norms  = |row| * |query|    → (N,) denominators
#       scores = dots / norms       → (N,) cosine in [-1, 1]
dots = matrix @ query_arr
matrix_norms = np.linalg.norm(matrix, axis=1)
query_norm = np.linalg.norm(query_arr)
scores = dots / (matrix_norms * query_norm)
print(f"score distribution: min={scores.min():.4f}  mean={scores.mean():.4f}  max={scores.max():.4f}\n")

# 5) top-k = indices of the highest scores; argsort ascending, reverse for descending
top_k = 3
top_idx = np.argsort(scores)[::-1][:top_k]
print(f"top-{top_k} matches:")
for rank, idx in enumerate(top_idx, start=1):
    emb = stored_embeddings[idx]
    print(f"#{rank}  score={scores[idx]:.4f}  {emb.chunk_id}  ({emb.source})")
    print(f"    {emb.text[:200].replace(chr(10), ' ')}...\n")

query vector dim: 384
first 5 values:   [0.07, -0.0714, 0.0336, -0.0404, 0.0157]

stored embeddings: 13 chunks (dim=384)

matrix shape: (13, 384)
query shape:  (384,)

score distribution: min=0.0050  mean=0.4199  max=0.6981

top-3 matches:
#1  score=0.6981  Large Language Models the New Scrum Masters#3  (Large Language Models the New Scrum Masters.md)
     which consists of professionals who work together to deliver increments of potentially shippable product  functionality during each Sprint. Traditionally, Scrum Masters have been human facilitators, ...

#2  score=0.6641  Large Language Models the New Scrum Masters#0  (Large Language Models the New Scrum Masters.md)
    Papers  RED Innovation: Using Scrum to Develop an  Agile Department  2024  Large Language Models, the New Scrum Masters  Large Language Models, the New Scrum Masters  Juan Couder  ortizcoj@my.erau.edu...

#3  score=0.6604  Large Language Models the New Scrum Masters#1  (Large Language Models the New Scrum Masters.md)


## Stage 5 — Answerer

- **Input:** a query string + the `RetrievalResult`s from stage 4.
- **Output:** an `Answer(query, text, model, chunks)` — the model's cited response plus the chunks it was given.
- **Backend:** OpenAI chat completions (default `gpt-4o-mini`). Reads `OPENAI_API_KEY` from `.env` (same as `ApiEmbedder`).
- **Prompt:** passed in as an argument so you can swap the framing without touching the answerer code. The default `DEFAULT_PROMPT` is a simple "answer-with-citations from context only" template; pass `prompt=` with your own `{query}` / `{chunks}` placeholders to change the behavior.

One entry point:
- `answer(query, chunks, prompt=DEFAULT_PROMPT, model=...)` — returns an `Answer`.

> Requires `OPENAI_API_KEY` in `.env`. The answerer formats each chunk with its `chunk_id` so the model can reference them in the response.

In [16]:
question = "How can large language models support a Scrum Master?"
candidates = retrieve(query=question, 
                      embedder=embedder, 
                      data_store=data_store, 
                      top_k=3)

prompt = """\
    You are a helpful research assistant. Answer the user's question using ONLY the provided context. \
    If the context does not contain the answer, say so plainly. \
    Cite the chunk_id of any fact you use, in square brackets like [chunk_id].

    Question: {query}

    Context:
    {chunks}
    """
response = answer(query=question, 
                  chunks=candidates,
                  prompt=prompt)

print(f"question: {response.query}\n")
print(f"answer ({response.model}):")
print(response.text)
print(f"\nchunks given to the model:")
for chunk in response.chunks:
    print(f"  - [{chunk.chunk_id}]  score={chunk.score:.4f}  source={chunk.source}")

question: How can large language models support a Scrum Master?

answer (gpt-4o-mini):
Large language models (LLMs) can support a Scrum Master by enhancing communication and collaboration within the team. They can analyze team communications, identify patterns, and offer insights to improve team dynamics. For example, LLMs can process messages from platforms like Slack or emails to detect trends in communication breakdowns or areas requiring clarification, enabling Scrum Masters to intervene proactively. Additionally, LLMs can assist in capturing the work of a Scrum Master during daily standup meetings, condensing information from team members' updates and highlighting any issues that may be blocking progress [3].

chunks given to the model:
  - [Large Language Models the New Scrum Masters#3]  score=0.6981  source=Large Language Models the New Scrum Masters.md
  - [Large Language Models the New Scrum Masters#0]  score=0.6641  source=Large Language Models the New Scrum Masters.md
  - [L

## Evaluation

Run the ground-truth eval suite (`eval/questions.json` — multi-choice + one-word questions, each anchored to a verbatim excerpt from the source PDF). Two metrics:

- **Retrieval accuracy** — does any top-k chunk contain the ground-truth excerpt?
- **Answer accuracy** — does the LLM's response contain the expected answer string?

The runner uses a tighter eval prompt than `DEFAULT_PROMPT` ("just the answer, no explanation") so the substring match is comparing apples to apples. Same `answer()` function — just a different `prompt=` argument.

In [17]:
from eval import run_eval

results = run_eval(top_k=5)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

c:\Users\User\Documents\bgu-ai\code\ai-rag-agents-course\rag\rag-demo\embedder\models.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimension = self._st.get_sentence_embedding_dimension()


[mc-001]  retrieval: PASS   answer: PASS
  Q: Which LLM does the paper use as a Scrum Master in the proposed approach?
  expected: GPT-3.5
  got:      GPT-3.5

[mc-002]  retrieval: PASS   answer: FAIL
  Q: How many student groups were used to evaluate the proposed procedure?
  expected: 6
  got:      Six groups.

[mc-003]  retrieval: PASS   answer: PASS
  Q: How long are the daily standup meetings described in the paper?
  expected: 15 minutes
  got:      15 minutes

[mc-004]  retrieval: PASS   answer: PASS
  Q: For which obstacle category did GPT provide zero successful solutions?
  expected: Need of new/different hardware
  got:      Contradicting or incorrect requirements and need of new/different hardware.

[mc-005]  retrieval: PASS   answer: PASS
  Q: What is the institutional affiliation of the paper's authors?
  expected: Embry-Riddle Aeronautical University
  got:      Embry-Riddle Aeronautical University

[ow-001]  retrieval: FAIL   answer: PASS
  Q: What percentage of the tot

### Inspect a single QA question — highlight the ground-truth span

Pick one question from `eval/questions.json`, retrieve top-k chunks, and highlight the verbatim answer excerpt inside each chunk using `spacy.displacy` (manual rendering, so no spaCy model download is required — just `pip install spacy`). Then run the answerer once and compare its response against the target answer.

In [18]:
import json
import re
from spacy import displacy

QUESTIONS_FILE = ROOT_DIR / "eval" / "questions.json"
questions = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))

# change the index to inspect a different question
question = questions[0]
print(f"[{question['id']}] {question['question']}")
print(f"expected: {question['answer']}\n")

retrieved = retrieve(
    query=question["question"],
    embedder=embedder,
    data_store=data_store,
    top_k=5,
)

excerpt = question["answer_location"]["excerpt"]
# whitespace-tolerant: PDF text often has line breaks inside the excerpt
pattern = re.compile(r"\s+".join(re.escape(p) for p in excerpt.split()), re.IGNORECASE)

rendered = []
for c in retrieved:
    m = pattern.search(c.text)
    ents = [{"start": m.start(), "end": m.end(), "label": "ANSWER"}] if m else []
    suffix = "  -- contains expected span" if m else ""
    rendered.append({
        "text": c.text,
        "ents": ents,
        "title": f"{c.chunk_id}  (score={c.score:.3f}){suffix}",
    })

displacy.render(
    rendered,
    style="ent",
    manual=True,
    jupyter=True,
    options={"colors": {"ANSWER": "#ffd166"}},
)

response = answer(query=question["question"], chunks=retrieved)
print(f"\nmodel answer: {response.text}")
print(f"target:       {question['answer']}")

[mc-001] Which LLM does the paper use as a Scrum Master in the proposed approach?
expected: GPT-3.5




model answer: The paper uses GPT-3.5 as the Large Language Model (LLM) in the proposed approach as a Scrum Master in a classroom setting [chunk_id: 5].
target:       GPT-3.5
